# Processing - Grouped Usage Data.ipynb

## Overview

This notebook loads **grouped metering data** from `data/grouped/` and produces interactive
charts for multi-tenant comparison, workspace-level breakdowns, and per-group trend analysis.

**Grouped data** means usage aggregated into time buckets and then split by a grouping
dimension — one set of time periods per group member (e.g. one line per tenant when grouped
by tenant). It is the right data source when you need to:

- **Compare tenants side-by-side** — who is the dominant consumer, and how has that changed?
- **Break down usage by workspace or instance** — which workspace is driving the most load?
- **Detect group-level anomalies** — did a specific tenant's usage spike unexpectedly?
- **Track share over time** — is one group's proportion growing at the expense of others?

If you need platform-wide totals without a group breakdown,
use **`Processing - Aggregated Usage Data.ipynb`** instead.
If you need individual record-level detail, use **`Processing - Raw Usage Data.ipynb`**.

---

### What this notebook produces

| # | Chart | What it answers |
|---|---|---|
| 5.1 | Usage over time — one line per group | How is each tenant/workspace/instance trending? |
| 5.2 | Total usage per group — horizontal bar | Who consumed the most over the full window? |
| 5.3 | Grouped bar chart — periods on X axis | How does per-period usage compare across groups? |
| 5.4 | Usage heatmap — groups × time buckets | Where are the hot spots across groups and time? |
| 5.5 | Percentage share — stacked area | Is one group's share growing or shrinking over time? |
| 5.6 | KPI cards | Total usage, active groups, peak day at a glance |
| 5.7 | Resource share — donut | What proportion of total usage does each group own? |
| 5.8 | Growth analysis — slope chart | Which groups grew or shrank the most (first vs. last period)? |
| 5.9 | Per-group statistics table | Mean, max, min, and total for every group in one view |

---

### Prerequisites

- This notebook runs out of the box using the **included sample data** — no deployment or API access needed.
- To use your own data, run **`Fetch - Usage Data.ipynb`** first to populate `data/grouped/<APP_DOMAIN>/<SERVICE_ID>/`, or point the configuration (Section 2) at an existing data directory.

All charts are fully interactive — hover for exact values, click legend items to toggle series,
drag to zoom, double-click to reset.


## 1. Imports

In [1]:
import json
import math as _math
import os
import pathlib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

## 2. Configuration

Load data path from environment variables or use defaults.
If you don't set `GROUPED_DATA_PATH` in `.env`, we'll use `data/grouped/<APP_DOMAIN>/<SERVICE_ID>/`.
To use custom data, edit `.env` and set `GROUPED_DATA_PATH`.

No secrets are loaded or printed here.

In [2]:
if pathlib.Path(".env").exists():
    load_dotenv(".env")
else:
    load_dotenv(".env.template")

service_id   = os.getenv("SERVICE_ID", "").strip()
app_domain  = os.getenv("APP_DOMAIN", "").strip()

# Use nested path structure: data/grouped/{APP_DOMAIN}/{SERVICE_ID}/
# APP_DOMAIN and SERVICE_ID are mandatory config values
if os.getenv("GROUPED_DATA_PATH"):
    data_path = os.getenv("GROUPED_DATA_PATH")  # Allow override via env var
else:
    data_path = f"data/grouped/{app_domain}/{service_id}"

DATA_DIR    = pathlib.Path(data_path)

json_files = list(DATA_DIR.glob("*.json"))

if json_files:
    print(f"✓ Using data path: {data_path}")
    print(f"✓ Found {len(json_files)} JSON file(s)")
else:
    print(f"⚠ No JSON files found in: {DATA_DIR.resolve()}")
    print(f"\nTo use a different path, edit .env and set:")
    print(f"  GROUPED_DATA_PATH=/path/to/your/data")
    print(f"\nExample: GROUPED_DATA_PATH=sample_data/grouped")

✓ Using data path: data/grouped/apps.gori-agent-hub.cp.fyre.ibm.com/servicebrokercore
✓ Found 9 JSON file(s)


## 3. Load Grouped Data

### Data format

Each JSON file has the shape returned by
`GET /metering/services/{serviceId}/usage/aggregated/{groupBy}`:

```json
{
  "params": {
    "metricId":   "users",
    "transform":  "max",
    "groupByHrs": 24,
    "groupBy":    "groupByTenant",
    "serviceId":  "cluster-as-a-service",
    "usageStart": 1783165639842,
    "usageEnd":   1785757639842
  },
  "totalGroups":          2,
  "totalPeriodsPerGroup": 8,
  "totalPeriodsCount":    16,
  "aggregatedMeteredUsagePeriods": [
    {
      "groupId":        "platform::",
      "groupTenantId":  "platform",
      "periodNumber":   1,
      "periodQuantity": 1,
      "periodStart":    1785110400000,
      "periodEnd":      1785196800000
    },
    {
      "groupId":        "::",
      "periodNumber":   1,
      "periodQuantity": 1,
      "periodStart":    1785110400000,
      "periodEnd":      1785196800000
    }
  ]
}
```

Each file is loaded as its own dataset. The `params` block is retained as metadata so charts
can label themselves with the correct metric, transform, bucket size, and grouping dimension.


In [3]:
datasets = []

for path in sorted(DATA_DIR.glob("*.json")):
    with open(path) as f:
        data = json.load(f)

    periods = data.get("aggregatedMeteredUsagePeriods", [])
    if not periods:
        print(f"  {path.name} — no periods, skipping")
        continue

    params      = data.get("params", {})
    df          = pd.DataFrame(periods)
    df["periodStart"]    = pd.to_datetime(df["periodStart"], unit="ms", utc=True)
    df["periodEnd"]      = pd.to_datetime(df["periodEnd"],   unit="ms", utc=True)
    df["periodQuantity"] = pd.to_numeric(df["periodQuantity"])

    # Friendly label for anonymous / empty tenant
    if "groupTenantId" in df.columns:
        df["groupTenantId"] = df["groupTenantId"].fillna("").replace("", "(anonymous)")

    # Friendly groupId label for charts (replace empty string tenant segment)
    if "groupId" in df.columns:
        df["groupLabel"] = df["groupId"].str.replace(r"^::", "(anonymous)::", regex=True)
    else:
        df["groupLabel"] = "(all)"

    group_by    = params.get("groupBy", "unknown")
    total_groups = data.get("totalGroups", df["groupId"].nunique() if "groupId" in df.columns else 1)

    datasets.append({
        "label":        path.stem,
        "metricId":     params.get("metricId", "unknown"),
        "transform":    params.get("transform", "unknown"),
        "groupByHrs":   params.get("groupByHrs", 0),
        "groupBy":      group_by,
        "totalGroups":  total_groups,
        "df":           df,
    })

    groups_preview = df["groupLabel"].unique().tolist() if "groupLabel" in df.columns else []
    print(
        f"  {path.name}\n"
        f"    metric={params.get('metricId')}  transform={params.get('transform')}  "
        f"groupByHrs={params.get('groupByHrs')}  groupBy={group_by}\n"
        f"    groups ({total_groups}): {groups_preview}"
    )

  last_30d_daily_avg_instances_by_instance.json
    metric=instances  transform=avg  groupByHrs=24  groupBy=groupByInstance
    groups (8): ['7ef3d047-2729-402c-a956-3292c88329a7:20260811-2007-0135-962b-8f2f8a078fa1:019ff26f-bf20-775f-a7fc-1704500c8fae', '0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1434-3899-16f9-16230953af1d:019ff66f-45a8-7114-b309-d5fa8fd3a965', 'platform:asaservice:019ff1ef-b43c-74e7-b483-d1fa606de79e', '0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1434-3899-16f9-16230953af1d:019ff75b-d8b0-75da-b0d8-5da6c8254df1', '0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1434-3899-16f9-16230953af1d:019ff75c-0850-71fc-91d1-b7f999237a7b', '0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1844-1996-964d-e84490b6e7d5:019ff74b-3438-7737-a1f2-c4bca2f63450', '0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1844-5538-161e-859079496e6a:019ff74b-cf0d-77fa-8d80-8a5f4135263e', 'ab475c00-63cc-4b75-abcf-9c4513505e31:20260812-1725-3257-36ec-2500ef2d54ce:019ff702-73fc-7608-9f7c-508c7ca3161f']
 

## 4. Summary table

| Column | Meaning |
|---|---|
| **Groups** | Distinct groups (tenants / workspaces / instances) returned |
| **Periods/group** | Buckets per group |
| **Bucket (h)** | Time window of each bucket in hours |
| **Window** | Date range covered by the dataset |
| **Min / Avg / Max** | `periodQuantity` statistics *across all groups and periods* |
| **Top group** | Group with the highest total (sum of `periodQuantity`) |

In [4]:
rows = []
for ds in datasets:
    df = ds["df"].sort_values("periodStart")
    q  = df["periodQuantity"]

    t_start = df["periodStart"].min().strftime("%b %d")
    t_end   = df["periodEnd"].max().strftime("%b %d")

    if "groupId" in df.columns:
        top_group = df.groupby("groupLabel")["periodQuantity"].sum().idxmax()
    else:
        top_group = "—"

    periods_per_group = (ds["totalGroups"] and (len(df) // ds["totalGroups"])) or len(df)

    rows.append({
        "Dataset":        ds["label"],
        "Metric":         ds["metricId"],
        "Transform":      ds["transform"],
        "Bucket (h)":     ds["groupByHrs"],
        "Group by":       ds["groupBy"],
        "Groups":         ds["totalGroups"],
        "Periods/group":  periods_per_group,
        "Window":         f"{t_start} – {t_end}",
        "Min":            round(q.min(), 2),
        "Avg":            round(q.mean(), 2),
        "Max":            round(q.max(), 2),
        "Top group":      top_group,
    })

pd.DataFrame(rows)

,Dataset,Metric,Transform,Bucket (h),Group by,Groups,Periods/group,Window,Min,Avg,Max,Top group
0,last_30d_daily_avg_instances_by_instance,instances,avg,24,groupByInstance,8,12,Aug 11 – Aug 25,1,1.00,1,7ef3d047-2729-402c-a956-3292c88329a7:20260811-...
1,last_30d_daily_avg_instances_by_tenant,instances,avg,24,groupByTenant,4,12,Aug 11 – Aug 25,1,1.00,1,7ef3d047-2729-402c-a956-3292c88329a7::
2,last_30d_daily_avg_instances_by_workspace,instances,avg,24,groupByWorkspace,6,12,Aug 11 – Aug 25,1,1.00,1,7ef3d047-2729-402c-a956-3292c88329a7:20260811-...
3,last_30d_daily_avg_users_by_instance,users,avg,24,groupByInstance,8,12,Aug 11 – Aug 25,1,1.00,1,7ef3d047-2729-402c-a956-3292c88329a7:20260811-...
4,last_30d_daily_avg_users_by_tenant,users,avg,24,groupByTenant,4,12,Aug 11 – Aug 25,1,1.00,1,7ef3d047-2729-402c-a956-3292c88329a7::
5,last_30d_daily_avg_users_by_workspace,users,avg,24,groupByWorkspace,6,12,Aug 11 – Aug 25,1,1.00,1,7ef3d047-2729-402c-a956-3292c88329a7:20260811-...
6,last_30d_daily_sum_api_calls_by_instance,api_calls,sum,24,groupByInstance,8,13,Aug 11 – Aug 25,104450,644639.26,745677,platform:asaservice:019ff1ef-b43c-74e7-b483-d1...
7,last_30d_daily_sum_api_calls_by_tenant,api_calls,sum,24,groupByTenant,4,13,Aug 11 – Aug 25,104450,1265403.00,3635083,0fd2a329-b354-44ad-90e7-16f00fa85948::
8,last_30d_daily_sum_api_calls_by_workspace,api_calls,sum,24,groupByWorkspace,6,13,Aug 11 – Aug 25,104450,854147.02,2178189,0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-...


## 5. Charts


### 5.1 Usage over time — line chart

### Computation

For the first loaded dataset, sorts periods chronologically and plots one line per group.
The Y axis label is derived from the `transform` field so it accurately reflects whether
values represent sums, averages, peaks, or floors.
Hover shows the period end timestamp and period number for precise identification.


In [5]:
fig_51 = None

if not datasets:
    print("No datasets loaded.")

else:
    # ── To show all datasets, replace the next 2 lines with:
    # ── for ds in datasets:
    ds = datasets[0]

    df = ds["df"].sort_values("periodStart").copy()

    # ------------------------------------------------------------
    # Y-axis label based on transform
    # ------------------------------------------------------------
    _y_labels = {
        "sum": "Total (sum per period)",
        "avg": "Average (mean per period)",
        "max": "Peak (max per period)",
        "min": "Floor (min per period)",
    }

    y_label = _y_labels.get(
        ds["transform"],
        f"Quantity ({ds['transform']})"
    )

    # ------------------------------------------------------------
    # Create short display labels for groups
    # Keep original groupLabel for hover information
    # ------------------------------------------------------------
    if "groupLabel" in df.columns:

        groups = df["groupLabel"].dropna().unique()

        group_mapping = {
            group: f"Group {i + 1}"
            for i, group in enumerate(groups)
        }

        df["groupDisplay"] = df["groupLabel"].map(group_mapping)

        n_groups = len(groups)

    else:
        df["groupDisplay"] = "All"
        n_groups = 1

    # ------------------------------------------------------------
    # Wrapped title
    # ------------------------------------------------------------
    title = (
        f"<b>{ds['metricId']}</b> — {y_label} · "
        f"{ds['groupByHrs']}h buckets"
        "<br>"
        f"<span style='font-size:14px'>"
        f"Grouped by: {ds['groupBy']} · "
        f"Source: {ds['label']}"
        f"</span>"
    )

    # ------------------------------------------------------------
    # Create chart
    # ------------------------------------------------------------
    fig_51 = px.line(
        df,
        x="periodStart",
        y="periodQuantity",
        color="groupDisplay",
        markers=True,
        title=title,

        labels={
            "periodStart": "Period start (UTC)",
            "periodQuantity": y_label,
            "groupDisplay": "Group",
        },

        custom_data=[
            "periodEnd",
            "periodNumber",
            "groupLabel",
        ],
    )

    # ------------------------------------------------------------
    # Hover information
    # ------------------------------------------------------------
    fig_51.update_traces(
        hovertemplate=(
            "<b>%{fullData.name}</b><br>"
            "Period: %{x}<br>"
            f"{y_label}: %{{y:.2f}}<br>"
            "Period end: %{customdata[0]}<br>"
            "Period number: %{customdata[1]}<br>"
            "Group ID: %{customdata[2]}"
            "<extra></extra>"
        )
    )

    # ------------------------------------------------------------
    # Legend calculation
    # ------------------------------------------------------------
    legend_rows = max(
        1,
        _math.ceil(n_groups / 3)
    )

    # ------------------------------------------------------------
    # Layout
    # ------------------------------------------------------------
    fig_51.update_layout(

        hovermode="x unified",

        plot_bgcolor="white",

        # More space at bottom for:
        #   1. rotated X-axis tick labels
        #   2. X-axis title
        #   3. gap
        #   4. Group legend
        margin=dict(
            t=110,
            b=220,
            l=80,
            r=50,
        ),

        # --------------------------------------------------------
        # Title
        # --------------------------------------------------------
        title=dict(
            x=0.5,
            xanchor="center",
            y=0.97,
            yanchor="top",
            font=dict(size=20),
        ),

        # --------------------------------------------------------
        # X-axis
        # --------------------------------------------------------
        xaxis=dict(
            title=dict(
                text="Period start (UTC)",
                standoff=20,
            ),
            tickangle=-45,
            automargin=True,
        ),

        # --------------------------------------------------------
        # Y-axis
        # --------------------------------------------------------
        yaxis=dict(
            title=y_label,
            automargin=True,
        ),

        # --------------------------------------------------------
        # Legend
        # --------------------------------------------------------
        legend=dict(
            title="Group",

            orientation="h",

            # IMPORTANT:
            # Move legend well below the X-axis title
            yanchor="top",
            y=-0.75,

            xanchor="center",
            x=0.5,

            entrywidth=120,
            entrywidthmode="pixels",

            font=dict(size=11),
        ),
    )

### Chart Guide

**Purpose:** Shows how each group's usage is trending over the query window.
The most fundamental view for grouped data — one line per tenant/workspace/instance.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Multi-series line chart |
| **X axis** | Period start timestamp (UTC) |
| **Y axis** | Usage quantity — label reflects the transform (sum/avg/max/min) |
| **Colour** | One line per group |

**Insights:** Diverging lines mean groups are growing at different rates.
A group whose line flattens while others rise is losing relative share.
> Showing the first dataset. To show all, change `ds = datasets[0]` to `for ds in datasets:`.


In [6]:
if fig_51 is not None:
    fig_51.show()


### 5.2 Total usage per group — horizontal bar

### Computation

Aggregates `periodQuantity` per group using the method that matches the dataset's `transform`:
`sum` → grand total, `avg` → mean of period averages, `max` → highest peak, `min` → lowest floor.
Groups are sorted ascending so the largest bar appears at the top.


In [7]:
fig_52 = None
if not datasets:
    print("No datasets loaded.")
else:
    # ── To show all datasets, replace the next 2 lines with:
    # ── for ds in datasets:  (and indent the block below)
    ds = datasets[0]
    df = ds["df"]
    if "groupLabel" in df.columns:
        # Aggregation per group must match the transform semantics:
        # sum  → grand total across all periods (meaningful)
        # avg  → mean of period averages per group
        # max  → highest peak seen per group
        # min  → lowest floor seen per group
        _agg_fn = {
            "sum": ("sum",  "Grand total across all periods"),
            "avg": ("mean", "Mean of period averages"),
            "max": ("max",  "Highest period peak"),
            "min": ("min",  "Lowest period floor"),
        }
        agg_method, agg_desc = _agg_fn.get(ds["transform"], ("sum", f"Total ({ds['transform']})"))
        group_totals = (
            df.groupby("groupLabel")["periodQuantity"]
            .agg(agg_method)
            .reset_index()
            .sort_values("periodQuantity", ascending=True)
        )
        group_totals.columns = ["group", "total"]
        # Short display label: first 8 chars + ellipsis; full value kept for hover
        group_totals["shortGroup"] = group_totals["group"].str[:8] + "…"
        title = (
            f"{ds['metricId']} — {agg_desc} per group\n"
            f"grouped by: {ds['groupBy']}  ·  source: {ds['label']}"
        )
        fig_52 = px.bar(
            group_totals,
            x="total",
            y="group",
            orientation="h",
            title=title,
            text="total",
            labels={"total": agg_desc, "group": "Group"},
            custom_data=["group"],
        )
        fig_52.update_traces(
            texttemplate="%{text:.2f}", textposition="outside", cliponaxis=False,
            hovertemplate="<b>%{customdata[0]}</b><br>" + agg_desc + ": %{x:,.2f}<extra></extra>",
        )
        # Override y-axis ticks: full UUID as tickval, short label as ticktext
        fig_52.update_yaxes(
            tickvals=group_totals["group"].tolist(),
            ticktext=group_totals["shortGroup"].tolist(),
        )
    else:
        print(f"  {ds['label']} — no groupId column, skipping");


### Chart Guide

**Purpose:** Ranks groups by total consumption across the full window at a glance.
The most direct answer to "who used the most?"

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Horizontal bar chart |
| **X axis** | Total usage quantity (aggregation matches transform) |
| **Y axis** | Group label |
| **Bar label** | Exact value printed outside each bar |

**Insights:** A dominant bar means one group accounts for a disproportionate share.
Very short bars may indicate idle or newly provisioned groups.
> Showing the first dataset. To show all, change `ds = datasets[0]` to `for ds in datasets:`.


In [8]:
if fig_52 is not None:
    fig_52.show()


### 5.3 Usage per period — grouped bar

### Computation

Plots `periodQuantity` per `(periodStart, group)` as a bar chart.
If the number of groups exceeds 8 (`STACK_THRESHOLD`), bars are **stacked** to avoid
overcrowding; otherwise they are **grouped** side by side for easier comparison.


In [9]:
fig_53 = None

if not datasets:
    print("No datasets loaded.")

else:
    # ── To show all datasets, replace the next 2 lines with:
    # ── for ds in datasets:
    ds = datasets[0]

    STACK_THRESHOLD = 8

    df = ds["df"].sort_values("periodStart").copy()

    # ------------------------------------------------------------
    # Check for group information
    # ------------------------------------------------------------
    if "groupLabel" not in df.columns:
        print(f"  {ds['label']} — no groupId column, skipping")

    else:

        # --------------------------------------------------------
        # Number of groups
        # --------------------------------------------------------
        n_groups = df["groupLabel"].nunique()

        # --------------------------------------------------------
        # Decide grouped vs stacked
        # --------------------------------------------------------
        barmode = (
            "stack"
            if n_groups > STACK_THRESHOLD
            else "group"
        )

        layout = (
            "stacked"
            if barmode == "stack"
            else "grouped"
        )

        # --------------------------------------------------------
        # Create short display names for the legend
        #
        # Original groupLabel is retained and shown in hover.
        # --------------------------------------------------------
        groups = df["groupLabel"].dropna().unique()

        group_mapping = {
            group: f"Group {i + 1}"
            for i, group in enumerate(groups)
        }

        df["groupDisplay"] = df["groupLabel"].map(group_mapping)

        # --------------------------------------------------------
        # Wrapped title
        # --------------------------------------------------------
        title = (
            f"<b>{ds['metricId']}</b> — "
            f"{ds['transform']} per period ({layout})"
            "<br>"
            f"<span style='font-size:14px'>"
            f"Grouped by: {ds['groupBy']} · "
            f"Source: {ds['label']}"
            "</span>"
        )

        # --------------------------------------------------------
        # Create bar chart
        # --------------------------------------------------------
        fig_53 = px.bar(
            df,
            x="periodStart",
            y="periodQuantity",
            color="groupDisplay",
            barmode=barmode,
            title=title,

            labels={
                "periodStart": "Period start (UTC)",
                "periodQuantity": f"Quantity ({ds['transform']})",
                "groupDisplay": "Group",
            },

            custom_data=[
                "groupLabel",
                "periodEnd",
            ],
        )

        # --------------------------------------------------------
        # Hover information
        # --------------------------------------------------------
        fig_53.update_traces(
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Period: %{x}<br>"
                f"Quantity: %{{y:.2f}}<br>"
                "Period end: %{customdata[1]}<br>"
                "Group ID: %{customdata[0]}"
                "<extra></extra>"
            )
        )

        # --------------------------------------------------------
        # Legend layout
        # --------------------------------------------------------
        # Allow approximately 3 legend items per row.
        legend_rows = max(
            1,
            _math.ceil(n_groups / 3)
        )

        # Reserve space for:
        #
        #   X-axis tick labels
        #   X-axis title
        #   spacing
        #   legend
        #
        bottom_margin = 170 + (legend_rows * 25)

        # --------------------------------------------------------
        # Layout
        # --------------------------------------------------------
        fig_53.update_layout(

            hovermode="x unified",

            plot_bgcolor="white",

            bargap=0.2,

            # ----------------------------------------------------
            # Margins
            # ----------------------------------------------------
            margin=dict(
                t=110,
                b=bottom_margin,
                l=80,
                r=50,
            ),

            # ----------------------------------------------------
            # Title
            # ----------------------------------------------------
            title=dict(
                x=0.5,
                xanchor="center",
                y=0.97,
                yanchor="top",
                font=dict(size=20),
            ),

            # ----------------------------------------------------
            # X-axis
            # ----------------------------------------------------
            xaxis=dict(
                title=dict(
                    text="Period start (UTC)",
                    standoff=20,
                ),

                tickangle=-45,

                automargin=True,
            ),

            # ----------------------------------------------------
            # Y-axis
            # ----------------------------------------------------
            yaxis=dict(
                title=f"Quantity ({ds['transform']})",

                automargin=True,
            ),

            # ----------------------------------------------------
            # Legend
            # ----------------------------------------------------
            legend=dict(
                title="Group",

                orientation="h",

                # IMPORTANT:
                # Put the legend well below the X-axis.
                yanchor="top",
                y=-0.48,

                xanchor="center",
                x=0.5,

                # Prevent individual legend entries
                # from becoming excessively wide.
                entrywidth=120,
                entrywidthmode="pixels",

                font=dict(size=11),
            ),
        )

### Chart Guide

**Purpose:** Shows how group usage compares period by period — useful for spotting
which groups spike or drop on specific dates.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Grouped bar (≤8 groups) or stacked bar (>8 groups) |
| **X axis** | Period start timestamp |
| **Y axis** | Usage quantity per period |
| **Colour** | One colour per group |

**Insights:** In grouped mode, bars of very different heights on the same date mean uneven load.
In stacked mode, a shrinking total bar means overall usage is declining.
> Showing the first dataset. To show all, change `ds = datasets[0]` to `for ds in datasets:`.


In [10]:
if fig_53 is not None:
    fig_53.show()


### 5.4 Usage heatmap — groups × time

### Computation

Pivots the dataset into a matrix of `(groupLabel, periodStart)` using `pivot_table`
with `aggfunc='sum'`. Missing cells are filled with zero.
Period start timestamps are formatted as `Mon DD` strings for readable column labels.


In [11]:
fig_54 = None
if not datasets:
    print("No datasets loaded.")
else:
    # ── To show all datasets, replace the next 2 lines with:
    # ── for ds in datasets:  (and indent the block below)
    ds = datasets[0]
    df = ds["df"].sort_values("periodStart")
    if "groupLabel" not in df.columns:
        print(f"  {ds['label']} — no groupId column, skipping")
    else:
        pivot = (
            df.pivot_table(
                index="groupLabel",
                columns="periodStart",
                values="periodQuantity",
                aggfunc="sum",
            )
            .fillna(0)
        )
        col_labels = [t.strftime("%b %d") for t in pivot.columns]
        full_y  = pivot.index.tolist()
        short_y = [s[:8] + "…" for s in full_y]
        title = (
            f"{ds['metricId']} — {ds['transform']} heatmap\n"
            f"grouped by: {ds['groupBy']}  ·  source: {ds['label']}"
        )
        fig_54 = px.imshow(
            pivot.values,
            x=col_labels,
            y=full_y,
            color_continuous_scale="Blues",
            text_auto=".2f",
            title=title,
            labels={"x": "Period start (UTC)", "y": "Group", "color": f"Qty ({ds['transform']})"},
            aspect="auto",
        )
        fig_54.update_xaxes(tickangle=-45)
        # Show short labels on y-axis; full label appears in hover via %{y}
        fig_54.update_yaxes(tickvals=full_y, ticktext=short_y)
        fig_54.update_traces(
            hovertemplate="<b>%{y}</b><br>Period: %{x}<br>Qty: %{z:.2f}<extra></extra>"
        )


### Chart Guide

**Purpose:** Shows the full usage landscape — every group × every time period — in one view.
Lets you spot both hot groups and hot periods simultaneously.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Annotated heatmap |
| **X axis** | Period start (formatted date) |
| **Y axis** | Group label |
| **Colour** | Usage quantity — darker = higher value |
| **Cell text** | Rounded value printed in each cell |

**Insights:** A dark row means one group consistently dominates.
A dark column means a particular period had unusually high usage across all groups.
> Showing the first dataset. To show all, change `ds = datasets[0]` to `for ds in datasets:`.


In [12]:
if fig_54 is not None:
    fig_54.show()


### 5.5 Percentage share by group — stacked area

### Computation

For each period, divides each group's `periodQuantity` by the period total to get a
percentage share. Uses `groupnorm='percent'` in Plotly to force a 0–100% stacked area.
Skipped if there is only one group (share would always be 100%).


In [13]:
fig_55 = None

if not datasets:
    print("No datasets loaded.")

else:
    # ── To show all datasets, replace the next 2 lines with:
    # ── for ds in datasets:
    ds = datasets[0]

    df = ds["df"].sort_values("periodStart").copy()

    # ------------------------------------------------------------
    # Check for group information
    # ------------------------------------------------------------
    if (
        "groupLabel" not in df.columns
        or df["groupLabel"].nunique() < 2
    ):
        print(
            f"  {ds['label']} — single group or no groupId, "
            "skipping share chart"
        )

    else:

        # --------------------------------------------------------
        # Calculate percentage share within each period
        # --------------------------------------------------------
        period_totals = (
            df.groupby("periodStart")["periodQuantity"]
            .transform("sum")
        )

        df["sharePct"] = (
            df["periodQuantity"]
            / period_totals.replace(0, np.nan)
            * 100
        )

        # --------------------------------------------------------
        # Create short display names for groups
        #
        # Keep the original groupLabel for hover information.
        # --------------------------------------------------------
        groups = df["groupLabel"].dropna().unique()

        group_mapping = {
            group: f"Group {i + 1}"
            for i, group in enumerate(groups)
        }

        df["groupDisplay"] = df["groupLabel"].map(group_mapping)

        n_groups = len(groups)

        # --------------------------------------------------------
        # Wrapped title
        # --------------------------------------------------------
        title = (
            f"<b>{ds['metricId']}</b> — group share (%) per period"
            "<br>"
            f"<span style='font-size:14px'>"
            f"Grouped by: {ds['groupBy']} · "
            f"Source: {ds['label']}"
            "</span>"
        )

        # --------------------------------------------------------
        # Create area chart
        # --------------------------------------------------------
        fig_55 = px.area(
            df,
            x="periodStart",
            y="sharePct",
            color="groupDisplay",
            title=title,

            labels={
                "periodStart": "Period start (UTC)",
                "sharePct": "Share (%)",
                "groupDisplay": "Group",
            },

            # Keep full group information for hover
            custom_data=[
                "groupLabel",
                "periodQuantity",
            ],

            # Forces 100% stacked area
            groupnorm="percent",
        )

        # --------------------------------------------------------
        # Hover information
        # --------------------------------------------------------
        fig_55.update_traces(
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Period: %{x}<br>"
                "Share: %{y:.2f}%<br>"
                "Quantity: %{customdata[1]:.2f}<br>"
                "Group ID: %{customdata[0]}"
                "<extra></extra>"
            )
        )

        # --------------------------------------------------------
        # Calculate legend layout
        # --------------------------------------------------------
        # Approximately 3 groups per row
        legend_rows = max(
            1,
            _math.ceil(n_groups / 3)
        )

        # Space for:
        #   - rotated X-axis tick labels
        #   - X-axis title
        #   - gap
        #   - legend
        bottom_margin = 170 + (legend_rows * 25)

        # --------------------------------------------------------
        # Layout
        # --------------------------------------------------------
        fig_55.update_layout(

            hovermode="x unified",

            # Keep chart between 0 and 100%
            yaxis=dict(
                range=[0, 100],
                title="Share (%)",
                automargin=True,
            ),

            plot_bgcolor="white",

            # ----------------------------------------------------
            # Margins
            # ----------------------------------------------------
            margin=dict(
                t=110,
                b=bottom_margin,
                l=80,
                r=50,
            ),

            # ----------------------------------------------------
            # Title
            # ----------------------------------------------------
            title=dict(
                x=0.5,
                xanchor="center",
                y=0.97,
                yanchor="top",
                font=dict(size=20),
            ),

            # ----------------------------------------------------
            # X-axis
            # ----------------------------------------------------
            xaxis=dict(
                title=dict(
                    text="Period start (UTC)",
                    standoff=20,
                ),

                tickangle=-45,

                automargin=True,
            ),

            # ----------------------------------------------------
            # Legend
            # ----------------------------------------------------
            legend=dict(
                title="Group",

                orientation="h",

                # Move legend below X-axis title
                yanchor="top",
                y=-0.48,

                xanchor="center",
                x=0.5,

                # Keep legend entries compact
                entrywidth=120,
                entrywidthmode="pixels",

                font=dict(size=11),
            ),
        );

### Chart Guide

**Purpose:** Shows whether any group's share of total usage is growing or shrinking over time —
a shift in proportions even when absolute values are stable.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | 100% stacked area chart |
| **X axis** | Period start timestamp |
| **Y axis** | Percentage share (0–100%) |
| **Colour** | One band per group |

**Insights:** A widening band means that group is taking a larger share over time.
If all bands stay roughly equal width, load is well-balanced across groups.
> Showing the first dataset. To show all, change `ds = datasets[0]` to `for ds in datasets:`.


In [14]:
if fig_55 is not None:
    fig_55.show()


### 5.6 Overview KPI cards

### Computation

Computes five summary statistics across all groups in the dataset:
total group count, largest group (name + total), smallest group (name + total),
count of idle groups (all-zero periods), and total number of reporting periods.
Each statistic is rendered as a `go.Indicator` number tile in a single row.


In [15]:
fig_56 = None
if not datasets:
    print("No datasets loaded.")
else:
    # ── To show all datasets, replace the next 2 lines with:
    # ── for ds in datasets:  (and indent the block below)
    ds = datasets[0]
    from plotly.subplots import make_subplots
    df = ds["df"]
    if "groupLabel" in df.columns:
        group_totals = df.groupby("groupLabel")["periodQuantity"].sum()
        n_groups     = len(group_totals)
        top_name     = group_totals.idxmax()
        top_val      = round(float(group_totals.max()), 2)
        bot_name     = group_totals.idxmin()
        bot_val      = round(float(group_totals.min()), 2)
        idle_groups  = int((group_totals == 0).sum())
        n_periods    = int(df["periodStart"].nunique())
        kpis = [
            {"title": "Total Groups",
            "value": n_groups,   "sub": ""},
            {"title": f"Largest · {top_name[:22]}",
            "value": top_val,    "sub": ds["transform"]},
            {"title": f"Smallest · {bot_name[:22]}",
            "value": bot_val,    "sub": ds["transform"]},
            {"title": "Idle Groups",
            "value": idle_groups, "sub": "all-zero periods"},
            {"title": "Reporting Periods",
            "value": n_periods,   "sub": "time buckets"},
        ]
        COLOURS = ["#636EFA", "#00CC96", "#EF553B", "#AB63FA", "#FFA15A"]
        fig_56 = make_subplots(
            rows=1, cols=len(kpis),
            specs=[[{"type": "domain"}] * len(kpis)],
        )
        for idx, kpi in enumerate(kpis):
            fig_56.add_trace(
                go.Indicator(
                    mode="number",
                    value=kpi["value"],
                    title={
                        "text": (
                            f"<b>{kpi['title']}</b><br>"
                            f"<span style='font-size:.75em;color:gray'>{kpi['sub']}</span>"
                        ),
                        "align": "center",
                    },
                    number={"font": {"size": 38, "color": COLOURS[idx]}, "valueformat": ".2f"},
                ),
                row=1, col=idx + 1,
            )
        fig_56.update_layout(
            height=200,
            title_text=f"{ds['metricId']} — overview · {ds['groupBy']} · {ds['label']}",
            paper_bgcolor="white",
            margin={"t": 60, "b": 10, "l": 10, "r": 10},
        )
    else:
        print(f"  {ds['label']} — no groupId, skipping KPI cards")


### Chart Guide

**Purpose:** Gives an instant summary of the dataset's scale and balance
without having to read a table.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | KPI indicator tiles (single row) |
| **Tile 1** | Total number of groups |
| **Tile 2** | Largest group name + total usage |
| **Tile 3** | Smallest group name + total usage |
| **Tile 4** | Count of idle groups (zero usage across all periods) |
| **Tile 5** | Number of distinct time periods in the dataset |

**Insights:** A high idle group count suggests many tenants/instances are provisioned
but not actively using the service. A large gap between largest and smallest
signals very uneven load distribution.
> Showing the first dataset. To show all, change `ds = datasets[0]` to `for ds in datasets:`.


In [16]:
if fig_56 is not None:
    fig_56.show()


### 5.7 Resource share — donut chart

### Computation

Sums `periodQuantity` per group across all periods to get each group's total.
Skipped automatically if there is only one group (a single 100% slice has no value).
The result is rendered as a donut with `hole=0.45`.


In [17]:
fig_57 = None

if not datasets:
    print("No datasets loaded.")

else:
    # ── To show all datasets, replace the next 2 lines with:
    # ── for ds in datasets:
    ds = datasets[0]
    df = ds["df"].copy()

    if "groupLabel" not in df.columns:
        print(f"  {ds['label']} — no groupLabel, skipping donut")

    else:

        # ------------------------------------------------------------
        # Aggregate by group
        # ------------------------------------------------------------
        group_totals = (
            df.groupby("groupLabel", dropna=False)["periodQuantity"]
            .sum()
            .reset_index()
        )

        group_totals.columns = ["group", "total"]

        # Remove null/empty groups
        group_totals = group_totals[
            group_totals["group"].notna()
            & (group_totals["group"].astype(str).str.strip() != "")
        ].copy()

        # ------------------------------------------------------------
        # Need at least 2 groups for a meaningful donut
        # ------------------------------------------------------------
        if group_totals["group"].nunique() < 2:

            if len(group_totals) > 0:
                group_name = group_totals["group"].iloc[0]
            else:
                group_name = "unknown"

            print(
                f"  {ds['label']} — only 1 group "
                f"({group_name}); donut skipped."
            )

        else:

            # --------------------------------------------------------
            # Create short labels for display
            # --------------------------------------------------------
            group_totals = group_totals.reset_index(drop=True)

            group_totals["groupLabel"] = [
                f"Group {i + 1}"
                for i in range(len(group_totals))
            ]

            # --------------------------------------------------------
            # Create donut
            # --------------------------------------------------------
            fig_57 = px.pie(
                group_totals,
                names="groupLabel",
                values="total",
                hole=0.45,

                title=(
                    f"{ds['metricId']} — share of total "
                    f"{ds['transform']} by group  ·  "
                    f"{ds['groupBy']}  ·  {ds['label']}"
                ),

                color_discrete_sequence=px.colors.qualitative.Plotly,

                custom_data=["group"],
            )

            # --------------------------------------------------------
            # Display only percentage on the donut
            # --------------------------------------------------------
            fig_57.update_traces(
                textinfo="percent",
                textposition="inside",

                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Total: %{value:.2f}<br>"
                    "Share: %{percent}"
                    "<extra></extra>"
                ),

                # Prevent text from becoming too small
                insidetextorientation="horizontal",

                # Don't allow labels to overflow
                automargin=True,
            )

            # --------------------------------------------------------
            # Layout
            # --------------------------------------------------------
            fig_57.update_layout(
                height=500,

                margin=dict(
                    l=40,
                    r=40,
                    t=100,
                    b=40,
                ),

                legend=dict(
                    title="Group",
                    orientation="v",
                    yanchor="middle",
                    y=0.5,
                    xanchor="left",
                    x=1.02,
                ),

                uniformtext=dict(
                    minsize=10,
                    mode="hide",
                ),
            );

### Chart Guide

**Purpose:** Shows each group's proportional share of total usage as an immediately
readable visual — the quickest way to see if one group is dominant.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Donut chart |
| **Segments** | One per group |
| **Size** | Proportional to the group's total usage across all periods |
| **Label** | Group name + percentage |

**Insights:** A segment taking more than 50% means one group dominates total consumption.
Many thin slivers mean load is broadly distributed — or many groups are nearly idle.
> Showing the first dataset. To show all, change `ds = datasets[0]` to `for ds in datasets:`.


In [18]:
if fig_57 is not None:
    fig_57.show()


### 5.8 Growth analysis — slope chart

### Computation

For each group, takes the **first** and **last** period values and draws a line between them.
The slope direction (↑ / ↓ / →) and delta are computed and shown in the legend.
Green = growth, red = decline, grey = stable (delta within ±0.01).


In [19]:
fig_58 = None

if not datasets:
    print("No datasets loaded.")

else:
    # ── To show all datasets, replace the next 2 lines with:
    # ── for ds in datasets:  (and indent the block below)

    ds = datasets[0]

    df = ds["df"].sort_values("periodStart").copy()

    # ------------------------------------------------------------
    # Validate data
    # ------------------------------------------------------------
    if "groupLabel" not in df.columns:
        print(
            f"  {ds['label']} — no groupId, skipping slope chart"
        )

    elif df["periodStart"].nunique() < 2:
        print(
            f"  {ds['label']} — only 1 period, "
            "slope chart needs ≥ 2 periods, skipping"
        )

    else:

        # --------------------------------------------------------
        # First and last dates
        # --------------------------------------------------------
        first_period = df["periodStart"].min()
        last_period = df["periodStart"].max()

        first_date = first_period.strftime("%b %d")
        last_date = last_period.strftime("%b %d")

        # --------------------------------------------------------
        # Create short display names for groups
        #
        # Original groupLabel is retained for hover information.
        # --------------------------------------------------------
        groups = (
            df["groupLabel"]
            .dropna()
            .unique()
            .tolist()
        )

        group_mapping = {
            group: f"Group {i + 1}"
            for i, group in enumerate(groups)
        }

        df["groupDisplay"] = df["groupLabel"].map(group_mapping)

        n_groups = len(groups)

        # --------------------------------------------------------
        # Create figure
        # --------------------------------------------------------
        fig_58 = go.Figure()

        for grp in groups:

            gdf = (
                df[df["groupLabel"] == grp]
                .sort_values("periodStart")
            )

            if gdf.empty:
                continue

            val_first = float(
                gdf["periodQuantity"].iloc[0]
            )

            val_last = float(
                gdf["periodQuantity"].iloc[-1]
            )

            delta = val_last - val_first

            # ----------------------------------------------------
            # Determine direction
            # ----------------------------------------------------
            if delta > 0.01:
                direction = "↑"
                colour = "#00CC96"

            elif delta < -0.01:
                direction = "↓"
                colour = "#EF553B"

            else:
                direction = "→"
                colour = "#888"

            display_name = group_mapping[grp]

            # ----------------------------------------------------
            # Add slope line
            # ----------------------------------------------------
            fig_58.add_trace(
                go.Scatter(
                    x=[
                        first_period,
                        last_period,
                    ],

                    y=[
                        val_first,
                        val_last,
                    ],

                    mode="lines+markers+text",

                    name=(
                        f"{display_name}  "
                        f"{direction} {delta:+.2f}"
                    ),

                    line={
                        "color": colour,
                        "width": 2.5,
                    },

                    marker={
                        "size": 11,
                        "color": colour,
                    },

                    text=[
                        f"{val_first:.2f}",
                        f"{val_last:.2f}",
                    ],

                    textposition=[
                        "middle left",
                        "middle right",
                    ],

                    # Keep full group ID in hover
                    customdata=[
                        [grp],
                        [grp],
                    ],

                    hovertemplate=(
                        "<b>%{fullData.name}</b><br>"
                        "Group ID: %{customdata[0]}<br>"
                        "%{x|%b %d %Y}: %{y:.3f}"
                        "<extra></extra>"
                    ),
                )
            )

        # --------------------------------------------------------
        # Legend spacing
        # --------------------------------------------------------
        # Approximately 3 legend items per row.
        legend_rows = max(
            1,
            _math.ceil(n_groups / 3)
        )

        # Space for:
        #   - X-axis tick labels
        #   - X-axis title
        #   - gap
        #   - legend
        bottom_margin = 170 + (legend_rows * 25)

        # --------------------------------------------------------
        # Title
        # --------------------------------------------------------
        title = (
            f"<b>{ds['metricId']}</b> — "
            f"growth: {first_date} → {last_date}"
            "<br>"
            f"<span style='font-size:14px'>"
            f"Grouped by: {ds['groupBy']} · "
            f"Source: {ds['label']}"
            "</span>"
        )

        # --------------------------------------------------------
        # Layout
        # --------------------------------------------------------
        fig_58.update_layout(

            title=dict(
                text=title,
                x=0.5,
                xanchor="center",
                y=0.97,
                yanchor="top",
                font=dict(size=20),
            ),

            # ----------------------------------------------------
            # X-axis
            # ----------------------------------------------------
            xaxis=dict(
                title=dict(
                    text="Period",
                    standoff=25,
                ),

                tickvals=[
                    first_period,
                    last_period,
                ],

                ticktext=[
                    first_date,
                    last_date,
                ],

                automargin=True,
            ),

            # ----------------------------------------------------
            # Y-axis
            # ----------------------------------------------------
            yaxis=dict(
                title=f"Quantity ({ds['transform']})",
                automargin=True,
            ),

            # ----------------------------------------------------
            # General appearance
            # ----------------------------------------------------
            height=500,

            plot_bgcolor="white",
            paper_bgcolor="white",

            hovermode="closest",

            # ----------------------------------------------------
            # Margins
            # ----------------------------------------------------
            margin=dict(
                t=110,
                b=bottom_margin,
                l=80,
                r=80,
            ),

            # ----------------------------------------------------
            # Legend
            # ----------------------------------------------------
            legend=dict(
                title="Group  (arrow = direction)",

                orientation="h",

                # Put legend clearly below X-axis title
                yanchor="top",
                y=-0.48,

                xanchor="center",
                x=0.5,

                entrywidth=140,
                entrywidthmode="pixels",

                font=dict(size=11),
            ),
        );

### Chart Guide

**Purpose:** Compares where each group started vs where it ended, making growth
and decline patterns immediately visible without reading a table.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Slope chart (two-point line per group) |
| **X axis** | Two points: first period date and last period date |
| **Y axis** | Usage quantity |
| **Green line** | Group grew over the window |
| **Red line** | Group shrank over the window |
| **Grey line** | Group was stable (change < 0.01) |

**Insights:** Many crossing lines indicate groups are swapping rank —
the dominant group at the start may not be dominant at the end.
> Showing the first dataset. To show all, change `ds = datasets[0]` to `for ds in datasets:`.


In [20]:
if fig_58 is not None:
    fig_58.show()


### 5.9 Per-group statistics table

### Computation

For each group in the first dataset, computes: mean, peak, minimum, count of active periods
(non-zero), count of idle periods (zero), and a trend arrow (↑/↓/→) based on comparing
the first and last period values. Rendered as a `pandas` DataFrame.

### Chart Guide

**Purpose:** Provides the full numerical summary for every group in one place —
the reference table to accompany all the visual charts above.

**Chart Attributes**

| Column | Meaning |
|---|---|
| **Avg** | Mean `periodQuantity` across all periods |
| **Peak** | Highest single period value |
| **Min** | Lowest single period value |
| **Active periods** | Count of periods with non-zero usage |
| **Idle periods** | Count of periods with zero usage |
| **Trend** | ↑ growing · ↓ declining · → stable (first vs last period) |

**Insights:** Groups with many idle periods may be provisioned but inactive.
A low average but high peak means usage is bursty — mostly quiet with occasional spikes.
> Showing the first dataset. To show all, change `ds = datasets[0]` to `for ds in datasets:`.



In [21]:
fig_59 = None
if not datasets:
    print("No datasets loaded.")
else:
    # ── To show all datasets, replace the next 2 lines with:
    # ── for ds in datasets:  (and indent the block below)
    ds = datasets[0]
    df = ds["df"].sort_values("periodStart")
    if "groupLabel" in df.columns:
        print(f"\n── {ds['metricId']} · {ds['groupBy']} · {ds['label']} ──")
        rows = []
        for grp, gdf in df.groupby("groupLabel"):
            gdf   = gdf.sort_values("periodStart")
            q     = gdf["periodQuantity"]
            first = float(q.iloc[0])
            last  = float(q.iloc[-1])
            delta = last - first
            trend = "↑" if delta > 0.01 else "↓" if delta < -0.01 else "→"
            rows.append({
                "Group":          grp,
                "Avg":            round(q.mean(), 3),
                "Peak":           round(q.max(),  3),
                "Min":            round(q.min(),  3),
                "Active periods": int((q > 0).sum()),
                "Idle periods":   int((q == 0).sum()),
                "Trend":          trend,
            })
        display(pd.DataFrame(rows).set_index("Group"))
    else:
        print(f"  {ds['label']} — no groupId column, skipping")



── instances · groupByInstance · last_30d_daily_avg_instances_by_instance ──


,Avg,Peak,Min,Active periods,Idle periods,Trend
Group,,,,,,
0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1434-3899-16f9-16230953af1d:019ff66f-45a8-7114-b309-d5fa8fd3a965,1.0,1,1,16,0,→
0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1434-3899-16f9-16230953af1d:019ff75b-d8b0-75da-b0d8-5da6c8254df1,1.0,1,1,16,0,→
0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1434-3899-16f9-16230953af1d:019ff75c-0850-71fc-91d1-b7f999237a7b,1.0,1,1,16,0,→
0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1844-1996-964d-e84490b6e7d5:019ff74b-3438-7737-a1f2-c4bca2f63450,1.0,1,1,16,0,→
0fd2a329-b354-44ad-90e7-16f00fa85948:20260812-1844-5538-161e-859079496e6a:019ff74b-cf0d-77fa-8d80-8a5f4135263e,1.0,1,1,16,0,→
7ef3d047-2729-402c-a956-3292c88329a7:20260811-2007-0135-962b-8f2f8a078fa1:019ff26f-bf20-775f-a7fc-1704500c8fae,1.0,1,1,16,0,→
ab475c00-63cc-4b75-abcf-9c4513505e31:20260812-1725-3257-36ec-2500ef2d54ce:019ff702-73fc-7608-9f7c-508c7ca3161f,1.0,1,1,16,0,→
platform:asaservice:019ff1ef-b43c-74e7-b483-d1fa606de79e,1.0,1,1,16,0,→
